# Notebook : 02_zone_remapping.ipynb
# TaaSim — Semaine 1 : Remapping Porto → Casablanca

CELLULE 1 — Bounding boxes des deux villes

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import folium
import json

spark = SparkSession.builder \
    .appName("TaaSim-Remapping") \
    .config("spark.driver.memory", "4g") \
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "taasim") \
    .config("spark.hadoop.fs.s3a.secret.key", "taasim123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print("✅ Spark prêt avec MinIO")
# ===============================
# Bounding boxes des villes
# ===============================

# Bounding box de Porto (Portugal)
# Source : OpenStreetMap / Geofabrik Portugal
PORTO_LON_MIN = -8.7327
PORTO_LON_MAX = -8.5539
PORTO_LAT_MIN = 41.0527
PORTO_LAT_MAX = 41.2370

# Bounding box de Casablanca (Maroc)
# Source : OpenStreetMap / Geofabrik Morocco
CASA_LON_MIN = -7.7500   # ← changé (était -7.7000) — plus à l'est
CASA_LON_MAX = -7.4500   # inchangé
CASA_LAT_MIN = 33.4500   # ← changé (était 33.4800) — remonté
CASA_LAT_MAX = 33.6500   # ← changé (était 33.6800) — descendu

# ===============================
# Affichage des bounding boxes
# ===============================
print("Porto → lon:", PORTO_LON_MIN, "→", PORTO_LON_MAX, "| lat:", PORTO_LAT_MIN, "→", PORTO_LAT_MAX)
print("Casa → lon:", CASA_LON_MIN, "→", CASA_LON_MAX, "| lat:", CASA_LAT_MIN, "→", CASA_LAT_MAX)

✅ Spark prêt avec MinIO
Porto → lon: -8.7327 → -8.5539 | lat: 41.0527 → 41.237
Casa → lon: -7.75 → -7.45 | lat: 33.45 → 33.65


In [2]:
!pip install -r /home/jovyan/work/notebooks/requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: '/home/jovyan/work/notebooks/requirements.txt'


CELLULE 2 — Fonctions de transformation

In [3]:
# %% [markdown]
# CELLULE 2 — Fonctions de transformation avec rotation géographique

# %%
import math

# ===============================
# Transformation linéaire standard
# ===============================

def transform_lon_linear(lon_porto):
    """Transformation linéaire simple longitude Porto → Casablanca"""
    ratio = (lon_porto - PORTO_LON_MIN) / (PORTO_LON_MAX - PORTO_LON_MIN)
    return CASA_LON_MIN + ratio * (CASA_LON_MAX - CASA_LON_MIN)

def transform_lat_linear(lat_porto):
    """Transformation linéaire simple latitude Porto → Casablanca"""
    ratio = (lat_porto - PORTO_LAT_MIN) / (PORTO_LAT_MAX - PORTO_LAT_MIN)
    return CASA_LAT_MIN + ratio * (CASA_LAT_MAX - CASA_LAT_MIN)

# ===============================
# Transformation avec rotation et étirement
# ===============================

def transform_point_with_stretch(lon_porto, lat_porto):
    """
    Transforme un point Porto → Casablanca avec :
    - Transformation linéaire de base
    - Étirement est-ouest (x1.3) car Casablanca est plus large que Porto
    - Léger étirement nord-sud (x1.1)
    - Correction des débordements
    """
    # 1. Transformation linéaire standard
    lon_casa = transform_lon_linear(lon_porto)
    lat_casa = transform_lat_linear(lat_porto)
    
    # 2. Centre de la bounding box Casablanca
    lon_center = (CASA_LON_MIN + CASA_LON_MAX) / 2  # environ -7.6
    lat_center = (CASA_LAT_MIN + CASA_LAT_MAX) / 2  # environ 33.565
    
    # 3. Facteurs d'étirement (tuning pour mieux couvrir Casablanca)
    stretch_east_west = 1.35      # Casablanca est plus étalée est-ouest
    stretch_north_south = 1.10    # Léger étirement nord-sud
    
    # 4. Appliquer l'étirement depuis le centre
    lon_casa = lon_center + (lon_casa - lon_center) * stretch_east_west
    lat_casa = lat_center + (lat_casa - lat_center) * stretch_north_south
    
    # 5. Correction : points qui sortent de la bounding box
    lon_casa = max(CASA_LON_MIN, min(CASA_LON_MAX, lon_casa))
    lat_casa = max(CASA_LAT_MIN, min(CASA_LAT_MAX, lat_casa))
    
    return lon_casa, lat_casa


def transform_point_with_rotation(lon_porto, lat_porto):
    """
    Transformation avec rotation géographique (alternative à l'étirement)
    Pour mieux aligner la côte de Porto (ouest) avec celle de Casablanca (nord-ouest)
    """
    # 1. Transformation linéaire standard
    lon_casa = transform_lon_linear(lon_porto)
    lat_casa = transform_lat_linear(lat_porto)
    
    # 2. Centre de rotation
    lon_center = (CASA_LON_MIN + CASA_LON_MAX) / 2
    lat_center = (CASA_LAT_MIN + CASA_LAT_MAX) / 2
    
    # 3. Décaler par rapport au centre
    dx = lon_casa - lon_center
    dy = lat_casa - lat_center
    
    # 4. Angle de rotation (en radians) = 15 degrés = 0.2618 rad
    # Rotation pour que les trajets côtiers de Porto (ouest) 
    # deviennent les trajets côtiers de Casablanca (nord-ouest)
    angle = 0.2618  # 15 degrés
    
    # 5. Appliquer la rotation
    dx_rot = dx * math.cos(angle) - dy * math.sin(angle)
    dy_rot = dx * math.sin(angle) + dy * math.cos(angle)
    
    # 6. Re-centrer
    lon_casa = lon_center + dx_rot
    lat_casa = lat_center + dy_rot
    
    # 7. Correction des débordements
    lon_casa = max(CASA_LON_MIN, min(CASA_LON_MAX, lon_casa))
    lat_casa = max(CASA_LAT_MIN, min(CASA_LAT_MAX, lat_casa))
    
    return lon_casa, lat_casa


# ===============================
# Fonction principale (utilise l'étirement par défaut)
# ===============================

def transform_point(lon_porto, lat_porto, method="stretch"):
    """
    Transforme un point Porto → Casablanca
    method: "stretch" (recommandé) ou "rotation" ou "linear"
    """
    if method == "stretch":
        return transform_point_with_stretch(lon_porto, lat_porto)
    elif method == "rotation":
        return transform_point_with_rotation(lon_porto, lat_porto)
    else:
        return transform_lon_linear(lon_porto), transform_lat_linear(lat_porto)


# ===============================
# Version UDF pour PySpark (avec étirement)
# ===============================

def remap_polyline(polyline_str):
    """Transforme tous les points GPS d'un trajet Porto → Casablanca avec étirement"""
    if polyline_str is None or polyline_str == "[]":
        return None
    try:
        points = json.loads(polyline_str)
        if len(points) < 2:
            return None
        remapped = []
        for pt in points:
            lon_porto, lat_porto = pt[0], pt[1]
            lon_casa, lat_casa = transform_point_with_stretch(lon_porto, lat_porto)
            remapped.append([round(lon_casa, 6), round(lat_casa, 6)])
        return json.dumps(remapped)
    except Exception as e:
        return None

# ===============================
# Test des fonctions
# ===============================

# Test sur les points caractéristiques de Porto
test_points = [
    (-8.6110, 41.1496),  # Centre de Porto
    (-8.6500, 41.1600),  # Ouest de Porto (côté mer)
    (-8.5700, 41.1400),  # Est de Porto
    (-8.6100, 41.2000),  # Nord de Porto
    (-8.6100, 41.1000),  # Sud de Porto
]

print("=" * 60)
print("Test des transformations Porto → Casablanca")
print("=" * 60)
print(f"{'Porto lon':>12} {'Porto lat':>12} → {'Casa lon (linear)':>16} {'Casa lat (linear)':>16} | {'Casa lon (stretch)':>16} {'Casa lat (stretch)':>16}")
print("-" * 85)

for lon_p, lat_p in test_points:
    lon_lin = transform_lon_linear(lon_p)
    lat_lin = transform_lat_linear(lat_p)
    lon_str, lat_str = transform_point_with_stretch(lon_p, lat_p)
    print(f"{lon_p:12.4f} {lat_p:12.4f} → {lon_lin:16.6f} {lat_lin:16.6f} | {lon_str:16.6f} {lat_str:16.6f}")

print("\n✅ Toutes les fonctions sont prêtes")
print("   La méthode 'stretch' est recommandée pour mieux couvrir Casablanca")

Test des transformations Porto → Casablanca
   Porto lon    Porto lat → Casa lon (linear) Casa lat (linear) | Casa lon (stretch) Casa lat (stretch)
-------------------------------------------------------------------------------------
     -8.6110      41.1496 →        -7.545805        33.555155 |        -7.526837        33.555670
     -8.6500      41.1600 →        -7.611242        33.566441 |        -7.615176        33.568085
     -8.5700      41.1400 →        -7.477013        33.544737 |        -7.450000        33.544211
     -8.6100      41.2000 →        -7.544128        33.609848 |        -7.524572        33.615833
     -8.6100      41.1000 →        -7.544128        33.501329 |        -7.524572        33.496462

✅ Toutes les fonctions sont prêtes
   La méthode 'stretch' est recommandée pour mieux couvrir Casablanca


CELLULE 3 — UDF PySpark pour le POLYLINE

In [4]:
# Le POLYLINE est un JSON array [[lon1,lat1], [lon2,lat2], ...]

# Enregistrer comme UDF Spark
remap_udf = F.udf(remap_polyline, StringType())

CELLULE 4 — Table de mapping des zones

In [5]:
import os

BASE_PATH = "/home/jovyan/work" if os.path.exists("/home/jovyan/work") else "."

zone_mapping_data = [
    # zones plus compactes, toutes dans la zone urbaine
    (1,  1,  "Ain Sebaa",           "industrial",   "medium", -7.5560, 33.6010, -7.5750, -7.5350, 33.5850, 33.6200, "2,5,12",    8.0),
    (2,  2,  "Al Fida",             "residential",  "high",   -7.6100, 33.5850, -7.6300, -7.5900, 33.5650, 33.6050, "1,3,7",     7.0),
    (3,  3,  "Anfa",                "commercial",   "medium", -7.6400, 33.5750, -7.6600, -7.6200, 33.5550, 33.5950, "2,8,10,16", 9.0),
    (4,  4,  "Ben M'Sick",          "residential",  "high",   -7.5850, 33.5550, -7.6050, -7.5650, 33.5350, 33.5750, "9,11,15",   7.0),
    (5,  5,  "Bernoussi",           "residential",  "high",   -7.5400, 33.6050, -7.5650, -7.5150, 33.5850, 33.6250, "1,12,14",   7.0),
    (6,  6,  "Bouskoura",           "residential",  "low",    -7.6350, 33.5300, -7.6600, -7.6100, 33.5200, 33.5500, "8,11",      10.0),
    (7,  7,  "El Fida-Mers Sultan", "commercial",   "high",   -7.5950, 33.5800, -7.6150, -7.5750, 33.5600, 33.6000, "2,9,13",    8.0),
    (8,  8,  "Hay Hassani",         "residential",  "high",   -7.6550, 33.5450, -7.6750, -7.6350, 33.5250, 33.5650, "3,6,10",    7.0),
    (9,  9,  "Hay Mohammadi",       "industrial",   "high",   -7.5650, 33.5700, -7.5850, -7.5450, 33.5500, 33.5900, "4,7,15",    7.0),
    (10, 10, "Maarif",              "commercial",   "medium", -7.6300, 33.5650, -7.6500, -7.6100, 33.5450, 33.5850, "3,8,16",    9.0),
    (11, 11, "Moulay Rachid",       "residential",  "high",   -7.5800, 33.5400, -7.6000, -7.5600, 33.5200, 33.5600, "4,6,15",    7.0),
    (12, 12, "Roches Noires",       "transit_hub",  "medium", -7.5700, 33.6000, -7.5900, -7.5500, 33.5800, 33.6200, "1,5,13",    8.0),
    (13, 13, "Sidi Belyout",        "commercial",   "high",   -7.6000, 33.5900, -7.6200, -7.5800, 33.5700, 33.6100, "7,12,16",   8.0),
    (14, 14, "Sidi Moumen",         "residential",  "high",   -7.5200, 33.5850, -7.5450, -7.4950, 33.5650, 33.6050, "5,15",      7.0),
    (15, 15, "Sidi Othmane",        "residential",  "high",   -7.5600, 33.5450, -7.5800, -7.5400, 33.5250, 33.5650, "4,9,11,14", 7.0),
    (16, 16, "Ain Diab",            "commercial",   "medium", -7.6650, 33.5900, -7.6850, -7.6450, 33.5700, 33.6100, "3,10,13",   10.0),
    # Porto 17-22
    (17, 1,  "Ain Sebaa",           "industrial",   "medium", -7.5560, 33.6010, -7.5750, -7.5350, 33.5850, 33.6200, "2,5,12",    8.0),
    (18, 5,  "Bernoussi",           "residential",  "high",   -7.5400, 33.6050, -7.5650, -7.5150, 33.5850, 33.6250, "1,12,14",   7.0),
    (19, 13, "Sidi Belyout",        "commercial",   "high",   -7.6000, 33.5900, -7.6200, -7.5800, 33.5700, 33.6100, "7,12,16",   8.0),
    (20, 3,  "Anfa",                "commercial",   "medium", -7.6400, 33.5750, -7.6600, -7.6200, 33.5550, 33.5950, "2,8,10,16", 9.0),
    (21, 10, "Maarif",              "commercial",   "medium", -7.6300, 33.5650, -7.6500, -7.6100, 33.5450, 33.5850, "3,8,16",    9.0),
    (16, 16, "Ain Diab",            "commercial",   "medium", -7.6550, 33.5900, -7.6750, -7.6450, 33.5800, 33.6000,  "3,10,13", 10.0),
]

schema_mapping = StructType([
    StructField("zone_porto",          IntegerType()),
    StructField("zone_id",             IntegerType()),
    StructField("zone_name",           StringType()),
    StructField("zone_type",           StringType()),
    StructField("population_density",  StringType()),
    StructField("centroid_lon",        DoubleType()),
    StructField("centroid_lat",        DoubleType()),
    StructField("bbox_lon_min",        DoubleType()),
    StructField("bbox_lon_max",        DoubleType()),
    StructField("bbox_lat_min",        DoubleType()),
    StructField("bbox_lat_max",        DoubleType()),
    StructField("adjacent_zones",      StringType()),
    StructField("base_fare_mad",       DoubleType()),
])

df_zones = spark.createDataFrame(zone_mapping_data, schema=schema_mapping)
df_zones.show(truncate=False)

# Sauvegarder CSV
os.makedirs(f"{BASE_PATH}/data", exist_ok=True)
df_zones.toPandas().to_csv(f"{BASE_PATH}/data/zone_mapping.csv", index=False)
print(f"zone_mapping.csv sauvegardé dans {BASE_PATH}/data/")

+----------+-------+-------------------+-----------+------------------+------------+------------+------------+------------+------------+------------+--------------+-------------+
|zone_porto|zone_id|zone_name          |zone_type  |population_density|centroid_lon|centroid_lat|bbox_lon_min|bbox_lon_max|bbox_lat_min|bbox_lat_max|adjacent_zones|base_fare_mad|
+----------+-------+-------------------+-----------+------------------+------------+------------+------------+------------+------------+------------+--------------+-------------+
|1         |1      |Ain Sebaa          |industrial |medium            |-7.556      |33.601      |-7.575      |-7.535      |33.585      |33.62       |2,5,12        |8.0          |
|2         |2      |Al Fida            |residential|high              |-7.61       |33.585      |-7.63       |-7.59       |33.565      |33.605      |1,3,7         |7.0          |
|3         |3      |Anfa               |commercial |medium            |-7.64       |33.575      |-7.66   

CELLULE 5 — Appliquer le remapping sur un échantillon

In [6]:
df = spark.read.csv("s3a://raw/porto-trips/train.csv", header=True, inferSchema=True)

# Filtrer les trajets valides
df_clean = df.filter(F.col("MISSING_DATA") == "False") \
             .filter(F.col("POLYLINE") != "[]") \
             .filter(F.col("POLYLINE").isNotNull())

# Sample
df_sample = df_clean.sample(0.02)

# Appliquer le remapping
df_remapped = df_sample.withColumn("POLYLINE_CASA", remap_udf(F.col("POLYLINE")))

# ── NOUVEAU : extraire TOUS les points du trajet ──────────────
def get_all_points(polyline_str):
    """Extrait tous les points GPS → liste de [lat, lon] pour folium"""
    if not polyline_str:
        return None
    try:
        pts = json.loads(polyline_str)
        if len(pts) < 2:
            return None
        return [[pt[1], pt[0]] for pt in pts]  # inverser lon,lat → lat,lon
    except:
        return None

get_all_points_udf = F.udf(get_all_points, ArrayType(ArrayType(DoubleType())))

# Construire le dataframe de visualisation
df_viz = df_remapped \
    .withColumn("route_casa", get_all_points_udf(F.col("POLYLINE_CASA"))) \
    .filter(F.col("route_casa").isNotNull()) \
    .select("TAXI_ID", "CALL_TYPE", "route_casa") \
    .limit(200)

points_pdf = df_viz.toPandas()
print(f"Trajets complets à visualiser : {len(points_pdf)}")
print(f"Exemple — nombre de points dans le 1er trajet : {len(points_pdf.iloc[0]['route_casa'])}")

Trajets complets à visualiser : 200
Exemple — nombre de points dans le 1er trajet : 32


CELLULE 6 — Carte Folium sur OpenStreetMap

In [7]:
colors_call = {"A": "blue", "B": "green", "C": "red"}

m = folium.Map(
    location=[33.5731, -7.5898],
    zoom_start=12,
    tiles="OpenStreetMap"
)

# Masque maritime : zone bleue transparente sur la mer
folium.Rectangle(
    bounds=[
        [33.60, -7.75],  # nord-ouest
        [33.70, -7.62]   # sud-est
    ],
    color="#88AAFF",
    fill=True,
    fill_opacity=0.3,
    weight=0,
    tooltip="Zone maritime (exclue)"
).add_to(m)

zones_pdf = df_zones.toPandas().drop_duplicates("zone_id")

for _, row in zones_pdf.iterrows():
    folium.Rectangle(
        bounds=[
            [row["bbox_lat_min"], row["bbox_lon_min"]],
            [row["bbox_lat_max"], row["bbox_lon_max"]]
        ],
        color="#1D9E75",
        fill=True,
        fill_opacity=0.08,
        weight=1,
        tooltip=f"Zone {row['zone_id']} — {row['zone_name']}<br>"
                f"Type: {row['zone_type']} | Fare: {row['base_fare_mad']} MAD"
    ).add_to(m)

    folium.Marker(
        location=[row["centroid_lat"], row["centroid_lon"]],
        icon=folium.DivIcon(
            html=f'<div style="font-size:11px;font-weight:bold;'
                 f'color:#085041;background:white;padding:2px 4px;'
                 f'border-radius:3px;border:1px solid #1D9E75">'
                 f'{row["zone_id"]}<br><span style="font-size:9px">'
                 f'{row["zone_name"]}</span></div>'
        )
    ).add_to(m)

# ── CORRECTION : trajets complets avec PolyLine ───────────────
for _, row in points_pdf.iterrows():
    route = row["route_casa"]          # ← changé de start_casa → route_casa
    if route is not None and len(route) >= 2:
        call_type = str(row["CALL_TYPE"])
        color = colors_call.get(call_type, "gray")

        # Trajet complet
        folium.PolyLine(
            locations=route,
            color=color,
            weight=1.5,
            opacity=0.6,
            tooltip=f"Taxi {row['TAXI_ID']} | Type {call_type} | {len(route)} pts"
        ).add_to(m)

        # Point de départ
        folium.CircleMarker(
            location=route[0],
            radius=4,
            color=color,
            fill=True,
            fill_opacity=1.0,
            tooltip=f"Départ — Taxi {row['TAXI_ID']}"
        ).add_to(m)

        # Point d'arrivée
        folium.CircleMarker(
            location=route[-1],
            radius=4,
            color="black",
            fill=True,
            fill_opacity=0.8,
            tooltip=f"Arrivée — Taxi {row['TAXI_ID']}"
        ).add_to(m)

legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background: white; padding: 12px; border: 1px solid #ccc;
     border-radius: 8px; font-size: 12px; min-width: 160px;">
  <b>TaaSim — Casablanca</b><br><br>
  <b>Type de course</b><br>
  <span style="color:blue">●</span> A — Dispatché<br>
  <span style="color:green">●</span> B — Station<br>
  <span style="color:red">●</span> C — Rue<br>
  <span style="color:black">●</span> Arrivée<br><br>
  <b>Zones</b><br>
  <span style="color:#1D9E75">■</span> Arrondissement Casa
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

os.makedirs(f"{BASE_PATH}/notebooks", exist_ok=True)
m.save(f"{BASE_PATH}/notebooks/casablanca_remapped_trips.html")
print("Carte sauvegardée")
m

Carte sauvegardée


CELLULE 7 — Validation du remapping

In [8]:
# Vérifier que tous les points transformés sont bien dans la bbox Casablanca
 
def validate_point(polyline_str):
    """Vérifie que tous les points sont dans la bbox Casa"""
    if not polyline_str:
        return True
    try:
        pts = json.loads(polyline_str)
        for pt in pts:
            lon, lat = pt[0], pt[1]
            if not (CASA_LON_MIN <= lon <= CASA_LON_MAX and
                    CASA_LAT_MIN <= lat <= CASA_LAT_MAX):
                return False
        return True
    except:
        return False
 
validate_udf = F.udf(validate_point, BooleanType())
 
df_validated = df_remapped.withColumn("valid", validate_udf(F.col("POLYLINE_CASA")))
n_total = df_validated.count()
n_valid = df_validated.filter(F.col("valid") == True).count()
n_invalid = n_total - n_valid
 
print(f"Total trajets testés : {n_total:,}")
print(f"Points dans bbox Casa : {n_valid:,} ({n_valid/n_total*100:.1f}%)")
print(f"Points hors bbox     : {n_invalid:,} ({n_invalid/n_total*100:.1f}%)")
 
if n_invalid / n_total > 0.02:
    print("ATTENTION : plus de 2% de points hors bbox — vérifier la transformation")
else:
    print("OK : remapping validé")

Total trajets testés : 34,189
Points dans bbox Casa : 34,189 (100.0%)
Points hors bbox     : 0 (0.0%)
OK : remapping validé


CELLULE 8 — Validation finale 

In [9]:
# ── Résumé final du zone_mapping généré ──────────────────────
zones_final = df_zones.toPandas().drop_duplicates("zone_id")

print("=== ZONE_MAPPING.CSV — RÉSUMÉ ===")
print(f"Nombre de zones Casablanca : {len(zones_final)}")
print(f"Types de zones : {zones_final['zone_type'].value_counts().to_dict()}")
print(f"Tarifs min/max : {zones_final['base_fare_mad'].min()} / {zones_final['base_fare_mad'].max()} MAD")
print()
print("Colonnes générées :")
for col in zones_final.columns:
    print(f"  ✓ {col}")
print()
print("Fichier prêt pour :")
print("  ✓ vehicle_gps_producer.py  — remapping coords")
print("  ✓ Flink Job 1              — assignment zone_id")
print("  ✓ Spark ETL Semaine 5      — features ML")

=== ZONE_MAPPING.CSV — RÉSUMÉ ===
Nombre de zones Casablanca : 16
Types de zones : {'residential': 8, 'commercial': 5, 'industrial': 2, 'transit_hub': 1}
Tarifs min/max : 7.0 / 10.0 MAD

Colonnes générées :
  ✓ zone_porto
  ✓ zone_id
  ✓ zone_name
  ✓ zone_type
  ✓ population_density
  ✓ centroid_lon
  ✓ centroid_lat
  ✓ bbox_lon_min
  ✓ bbox_lon_max
  ✓ bbox_lat_min
  ✓ bbox_lat_max
  ✓ adjacent_zones
  ✓ base_fare_mad

Fichier prêt pour :
  ✓ vehicle_gps_producer.py  — remapping coords
  ✓ Flink Job 1              — assignment zone_id
  ✓ Spark ETL Semaine 5      — features ML
